# 대청댐 조류경보 예측 모델링 노트북

반영 확인: 2026-05-02 / 브리핑용 정리본

이 노트북은 **AI 모델링 전담 파트** 산출물입니다. 전처리팀이 만든 최종 모델 입력 테이블을 받아서 모델 학습, 평가, 해석, 시나리오 후처리 결과를 생성합니다.

```text
전처리 완료 데이터
→ 시간 기준 train/validation split
→ feature/target 분리
→ Gradient Boosting 후보 모델 학습
→ 회귀·분류 성능 비교
→ best model 선택
→ 예측 결과 저장
→ SHAP/feature importance 원인 추출
→ 시나리오 결과 CSV/JSON 저장
```

이 노트북에서 하지 않는 것:

- 원천 데이터 수집
- 수질·수문·기상 데이터 병합
- 결측치/이상치 처리
- lag/rolling/누적강수량 feature 생성
- target label 생성
- 보고서/PPT/브리핑 문장 생성


## 0. 브리핑용 핵심 구조

이 노트북은 아래 세 덩어리로 읽으면 됩니다.

1. **Config**: 경로, 컬럼명, target, threshold, 시나리오 feature group 설정
2. **함수 정의**: 로드, split, 학습, 평가, 저장, 설명, 시나리오 후처리 함수
3. **실행 파이프라인**: 실제로 위에서 아래로 실행하는 모델링 흐름

실행 파이프라인은 `## 5. 실행 파이프라인` 섹션부터 시작합니다.


## 1. 입력 데이터 가정

입력 파일은 전처리와 feature engineering이 완료된 최종 모델 입력 테이블입니다.

```text
data/processed/model_input/algae_model_input.csv
```

선택 schema 파일:

```text
data/processed/model_input/model_input_schema.csv
```

schema가 있으면 `role == "feature"`인 컬럼만 모델 입력으로 사용합니다. schema가 없으면 `DROP_COLUMNS`를 제외한 컬럼을 feature 후보로 사용하고, 누수 의심 키워드를 검사합니다.


## 2. 라이브러리 Import


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)


## 3. Config

실제 데이터셋에 맞게 수정할 값은 이 셀에 모읍니다.

특히 확인할 값:

- `MODEL_INPUT_PATH`
- `SCHEMA_PATH`
- `DATE_COLUMN`, `SITE_COLUMN`
- `REGRESSION_TARGET`, `CLASSIFICATION_TARGET`
- `DROP_COLUMNS`
- `REGRESSION_TARGET_TRANSFORM`
- `SCENARIO_FEATURE_GROUPS`


In [ ]:
# ============================================================
# 경로 설정
# ============================================================
MODEL_INPUT_PATH = Path("data/processed/model_input/algae_model_input.csv")
SCHEMA_PATH = Path("data/processed/model_input/model_input_schema.csv")  # 선택 입력
OUTPUT_DIR = Path("artifacts/models")
PREDICTION_DIR = Path("artifacts/predictions")
METRIC_DIR = Path("artifacts/metrics")
EXPLAIN_DIR = Path("artifacts/explain")
SCENARIO_DIR = Path("artifacts/scenario")

# 최종 산출물 파일명입니다. 다음 에이전트가 이 파일들을 입력으로 사용한다고 가정합니다.
REGRESSION_PREDICTION_FILE = "regression_predictions.csv"
CLASSIFICATION_PREDICTION_FILE = "classification_predictions.csv"
REGRESSION_METRIC_FILE = "regression_metrics.json"
CLASSIFICATION_METRIC_FILE = "classification_metrics.json"
FEATURE_IMPORTANCE_FILE = "feature_importance.csv"
SHAP_TOP_REASONS_FILE = "shap_top_reasons.csv"
SCENARIO_RESULT_FILE = "scenario_results.csv"
SCENARIO_LLM_INPUT_FILE = "scenario_input_for_llm.json"
REGRESSION_MODEL_FILE = "regression_model.pkl"
CLASSIFICATION_MODEL_FILE = "classification_model.pkl"
PREPROCESSING_PIPELINE_FILE = "preprocessing_pipeline.pkl"

# ============================================================
# 컬럼 설정
# ============================================================
DATE_COLUMN = "sample_date"      # 수정 필요: 전처리팀 산출물에 맞게 수정
SITE_COLUMN = "site"             # 수정 필요: 전처리팀 산출물에 맞게 수정
SPLIT_COLUMN = "split"           # 선택 입력: 있으면 train/valid split에 우선 사용
FORECAST_HORIZON = "t_plus_7"    # 다음 에이전트 전달용 메타 정보

REGRESSION_TARGET = "target_log_cells_t_plus_7"  # 수정 필요: 실제 회귀 target 컬럼명으로 수정
CLASSIFICATION_TARGET = "target_alert_t_plus_7"  # 수정 필요: 실제 분류 target 컬럼명으로 수정

# 모델 입력에서 제외할 컬럼입니다.
# 정답 컬럼, 식별자, 날짜, 분할/관리 컬럼, 미래 정답, 운영 라벨은 입력 변수로 쓰면 데이터 누수가 생길 수 있습니다.
# 가장 안전한 방식은 아래 제외 컬럼 목록보다 schema 기반 허용 목록을 우선 사용하는 것입니다.
DROP_COLUMNS = [
    DATE_COLUMN,
    SITE_COLUMN,
    SPLIT_COLUMN,
    REGRESSION_TARGET,
    CLASSIFICATION_TARGET,
    "target_cells_t_plus_7",       # 선택 입력: 미래 세포수 정답
    "alert_stage",                 # 선택 입력: 현재/미래 여부가 불명확한 운영 라벨은 기본 제외
    "raw_cells",                   # 선택 입력: 현재값인지 미래값인지 불명확하면 기본 제외
]

# ============================================================
# 입력 변수 안전 설정
# ============================================================
# 가장 안전한 방식은 전처리팀이 model_input_schema.csv에서 입력 변수로 승인한 컬럼만 사용하는 것입니다.
# schema 파일이 없으면 제외 컬럼 목록 기반으로 입력 변수 후보를 만들고, 누수 의심 키워드를 자동 검사합니다.
SCHEMA_COLUMN_NAME = "column_name"
SCHEMA_ROLE_NAME = "role"
FEATURE_ROLE_VALUE = "feature"

FORBIDDEN_FEATURE_KEYWORDS = [
    "target",
    "future",
    "next",
    "t_plus",
    "label",
    "answer",
    "ground_truth",
]

# 문자열/범주형 입력 변수는 전처리팀이 미리 인코딩했다고 가정합니다.
# 따라서 모델링 노트북에서는 입력 변수가 숫자형인지 검사만 합니다.
REQUIRE_NUMERIC_FEATURES = True

# 비교할 그래디언트 부스팅 후보 모델입니다.
# "auto"는 설치된 LightGBM, XGBoost, sklearn HistGradientBoosting을 모두 비교합니다.
# 특정 모델만 비교하고 싶으면 후보 모델 목록을 직접 수정합니다.
REGRESSION_MODEL_CANDIDATES = ["lightgbm", "xgboost", "hist_gradient_boosting"]
CLASSIFICATION_MODEL_CANDIDATES = ["lightgbm", "xgboost", "hist_gradient_boosting"]
MODEL_SELECTION_METRIC_REGRESSION = "rmse"      # 낮을수록 좋음
MODEL_SELECTION_METRIC_CLASSIFICATION = "recall"  # 조류경보는 미탐 방지를 위해 recall 우선

# ============================================================
# 모델링 설정
# ============================================================
RANDOM_STATE = 42
VALID_SIZE_RATIO = 0.2
PROBABILITY_THRESHOLD = 0.5  # 기본 threshold. validation 기반 후보 탐색 후 조정 가능
ALERT_CELL_THRESHOLD = 1000  # 수정 필요: 공식 기준 확인 후 config에서 확정

# 회귀 정답값 변환 방식입니다.
# 과제 기본 가정은 log1p(세포수)로 학습한 뒤 expm1로 원 단위를 복원하는 방식입니다.
# 전처리팀이 log10(세포수 + 1)을 사용했다면 "log10_plus_1"로 바꾸세요.
REGRESSION_TARGET_TRANSFORM = "log1p"  # one of: "log1p", "log10_plus_1", "none"

# 기준값 후보 탐색 설정입니다.
# 주의: 최종 운영 기준값은 별도 검증 기간에서 정하고, 테스트 성능 부풀리기에 사용하면 안 됩니다.
THRESHOLD_CANDIDATES = [round(x, 2) for x in np.arange(0.10, 0.91, 0.05)]
MIN_PRECISION_FOR_THRESHOLD = 0.30

# ============================================================
# 시나리오 후처리 설정
# ============================================================
# 아래 기준값은 임시값입니다. 실제 데이터 분포와 검증 결과에 따라 조정해야 합니다.
HIGH_RISK_THRESHOLD = 0.80
MEDIUM_RISK_THRESHOLD = 0.40

SCENARIO_FEATURE_GROUPS = {
    "stagnation": [
        "residence_proxy",
        "outflow",
        "outflow_7d",
        "low_wind_days",
        "wind_mean_7d",
        "storage",
    ],
    "rainfall_inflow": [
        "rainfall_3d",
        "rainfall_7d",
        "rainfall_14d",
        "inflow",
        "inflow_7d",
    ],
    "growth_condition": [
        "water_temp",
        "air_temp_mean_7d",
        "ground_temp_mean_7d",
        "solar_sum_7d",
        "ph",
    ],
    "past_bloom": [
        "current_cells",
        "prev_cells",
        "delta_cells",
        "growth_rate_cells",
    ],
}

SCENARIO_ACTION_CATEGORY = {
    "수체 정체·성층 위험 시나리오": "물순환설비 가동 검토",
    "강우 이후 영양염류 유입 시나리오": "추가 채수 및 현장 확인",
    "고온·고일사 성장 촉진 시나리오": "모니터링 강화",
    "과거 증식 추세 지속 시나리오": "추가 채수 및 현장 확인",
    "복합 고위험 시나리오": "관계기관 공유 필요",
    "관찰 강화 시나리오": "모니터링 강화",
    "일반 안정 시나리오": "일반 관찰 유지",
}


## 4. 함수 정의

아래 함수들은 모델링 파이프라인을 구성하는 부품입니다. 실제 실행은 `5. 실행 파이프라인`부터 진행합니다.


### 4.1 데이터 로드


In [ ]:
def load_model_input(path: Path) -> pd.DataFrame:
    """전처리 완료 모델 입력 테이블을 읽습니다.

    이 함수는 전처리나 feature engineering을 수행하지 않습니다.
    이미 완성된 모델 입력 테이블을 읽기만 합니다.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"Model input file not found: {path}. "
            "전처리팀의 최종 산출물 경로를 MODEL_INPUT_PATH에 맞게 수정하세요."
        )

    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    return pd.read_csv(path)


### 4.2 Feature / Target 분리

목적은 데이터 누수 방지입니다. 미래 정답 컬럼, target 컬럼, split/meta 컬럼이 feature로 들어가지 않도록 검사합니다.


In [ ]:
def load_model_input_schema(schema_path: Path) -> pd.DataFrame | None:
    """선택 입력값인 모델 입력 schema 파일을 읽습니다.

    schema는 feature engineering이 아니라, 전처리팀과 모델링팀 사이에서
    어떤 컬럼을 feature로 사용할지 명시하는 안전 계약입니다.
    """
    if not schema_path.exists():
        return None
    return pd.read_csv(schema_path)


def get_feature_columns_from_schema(
    df: pd.DataFrame,
    schema_df: pd.DataFrame,
    column_name_col: str = "column_name",
    role_col: str = "role",
    feature_role_value: str = "feature",
) -> list[str]:
    required_cols = {column_name_col, role_col}
    missing = required_cols - set(schema_df.columns)
    if missing:
        raise KeyError(f"schema 파일에 필수 컬럼이 없습니다: {sorted(missing)}")

    feature_columns = (
        schema_df.loc[schema_df[role_col].eq(feature_role_value), column_name_col]
        .dropna()
        .astype(str)
        .tolist()
    )

    missing_in_df = [col for col in feature_columns if col not in df.columns]
    if missing_in_df:
        raise KeyError(f"schema에서 feature로 승인된 컬럼이 모델 입력 테이블에 없습니다: {missing_in_df}")

    return feature_columns


def get_feature_columns_by_drop_rule(df: pd.DataFrame, drop_columns: list[str]) -> list[str]:
    existing_drop_columns = [col for col in drop_columns if col in df.columns]
    feature_columns = [col for col in df.columns if col not in existing_drop_columns]
    return feature_columns


def check_leakage_columns(
    feature_columns: list[str],
    forbidden_keywords: list[str],
) -> None:
    suspicious = [
        col for col in feature_columns
        if any(keyword in col.lower() for keyword in forbidden_keywords)
    ]
    if suspicious:
        raise ValueError(
            "누수 의심 컬럼이 feature에 포함되어 있습니다. "
            f"schema 또는 DROP_COLUMNS를 확인하세요: {suspicious}"
        )


def check_required_targets(
    df: pd.DataFrame,
    regression_target: str,
    classification_target: str,
) -> None:
    missing = [col for col in [regression_target, classification_target] if col not in df.columns]
    if missing:
        raise KeyError(f"모델 입력 테이블에 target 컬럼이 없습니다: {missing}")


def check_numeric_features(df: pd.DataFrame, feature_columns: list[str]) -> None:
    non_numeric = [col for col in feature_columns if not pd.api.types.is_numeric_dtype(df[col])]
    if non_numeric:
        raise TypeError(
            "모델링 노트북은 전처리 완료 feature table을 입력으로 받습니다. "
            "문자형/categorical feature는 전처리팀에서 encoding 후 전달해야 합니다. "
            f"Non-numeric feature columns: {non_numeric}"
        )


def get_feature_columns(
    df: pd.DataFrame,
    drop_columns: list[str],
    schema_path: Path | None = None,
) -> list[str]:
    schema_df = load_model_input_schema(schema_path) if schema_path is not None else None

    if schema_df is not None:
        feature_columns = get_feature_columns_from_schema(
            df,
            schema_df,
            column_name_col=SCHEMA_COLUMN_NAME,
            role_col=SCHEMA_ROLE_NAME,
            feature_role_value=FEATURE_ROLE_VALUE,
        )
    else:
        feature_columns = get_feature_columns_by_drop_rule(df, drop_columns)

    check_required_targets(df, REGRESSION_TARGET, CLASSIFICATION_TARGET)
    check_leakage_columns(feature_columns, FORBIDDEN_FEATURE_KEYWORDS)

    if REQUIRE_NUMERIC_FEATURES:
        check_numeric_features(df, feature_columns)

    return feature_columns


def split_features_targets(
    df: pd.DataFrame,
    feature_columns: list[str],
    regression_target: str,
    classification_target: str,
) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    check_required_targets(df, regression_target, classification_target)

    x = df[feature_columns]
    y_reg = df[regression_target]
    y_cls = df[classification_target]
    return x, y_reg, y_cls


### 4.3 시간 기준 Train / Validation Split


In [ ]:
def time_based_train_valid_split(
    df: pd.DataFrame,
    date_column: str,
    split_column: str | None = None,
    valid_size_ratio: float = 0.2,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if split_column and split_column in df.columns:
        train_df = df[df[split_column] == "train"].copy()
        valid_df = df[df[split_column].isin(["valid", "validation"])].copy()
        if len(train_df) == 0 or len(valid_df) == 0:
            raise ValueError("split column exists, but train/valid rows are empty. Check split labels.")
        return train_df, valid_df

    if date_column not in df.columns:
        raise KeyError(f"Date column not found: {date_column}")

    sorted_df = df.copy()
    sorted_df[date_column] = pd.to_datetime(sorted_df[date_column])
    sorted_df = sorted_df.sort_values(date_column).reset_index(drop=True)

    split_idx = int(len(sorted_df) * (1 - valid_size_ratio))
    train_df = sorted_df.iloc[:split_idx].copy()
    valid_df = sorted_df.iloc[split_idx:].copy()
    return train_df, valid_df


### 4.4 후보 모델 정의

LightGBM, XGBoost, HistGradientBoosting을 후보로 비교합니다. 설치되지 않은 외부 라이브러리는 자동으로 건너뜁니다.


In [ ]:
def _expand_model_candidates(candidates: list[str]) -> list[str]:
    if "auto" in candidates:
        return ["lightgbm", "xgboost", "hist_gradient_boosting"]
    return candidates


def build_regression_model_candidates(random_state: int = 42) -> dict[str, Any]:
    """사용 가능한 Gradient Boosting 회귀 후보 모델을 모두 만듭니다.

    이 함수는 모델 하나를 자동으로 선택하지 않습니다.
    validation 성능 지표로 best model을 고를 수 있도록 모든 후보를 반환합니다.
    """
    candidates = {}
    requested = _expand_model_candidates(REGRESSION_MODEL_CANDIDATES)

    if "lightgbm" in requested:
        try:
            from lightgbm import LGBMRegressor
            candidates["lightgbm"] = LGBMRegressor(
                objective="regression",
                n_estimators=500,
                learning_rate=0.03,
                num_leaves=31,
                random_state=random_state,
            )
        except ImportError:
            pass

    if "xgboost" in requested:
        try:
            from xgboost import XGBRegressor
            candidates["xgboost"] = XGBRegressor(
                objective="reg:squarederror",
                n_estimators=500,
                learning_rate=0.03,
                max_depth=5,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=random_state,
            )
        except ImportError:
            pass

    if "hist_gradient_boosting" in requested:
        candidates["hist_gradient_boosting"] = HistGradientBoostingRegressor(
            learning_rate=0.03,
            max_iter=500,
            random_state=random_state,
        )

    if not candidates:
        raise ImportError("사용 가능한 회귀 모델 후보가 없습니다. lightgbm, xgboost 또는 scikit-learn 설치를 확인하세요.")
    return candidates


def build_classification_model_candidates(random_state: int = 42) -> dict[str, Any]:
    """사용 가능한 Gradient Boosting 분류 후보 모델을 모두 만듭니다."""
    candidates = {}
    requested = _expand_model_candidates(CLASSIFICATION_MODEL_CANDIDATES)

    if "lightgbm" in requested:
        try:
            from lightgbm import LGBMClassifier
            candidates["lightgbm"] = LGBMClassifier(
                objective="binary",
                n_estimators=500,
                learning_rate=0.03,
                num_leaves=31,
                random_state=random_state,
            )
        except ImportError:
            pass

    if "xgboost" in requested:
        try:
            from xgboost import XGBClassifier
            candidates["xgboost"] = XGBClassifier(
                objective="binary:logistic",
                n_estimators=500,
                learning_rate=0.03,
                max_depth=5,
                subsample=0.8,
                colsample_bytree=0.8,
                eval_metric="logloss",
                random_state=random_state,
            )
        except ImportError:
            pass

    if "hist_gradient_boosting" in requested:
        candidates["hist_gradient_boosting"] = HistGradientBoostingClassifier(
            learning_rate=0.03,
            max_iter=500,
            random_state=random_state,
        )

    if not candidates:
        raise ImportError("사용 가능한 분류 모델 후보가 없습니다. lightgbm, xgboost 또는 scikit-learn 설치를 확인하세요.")
    return candidates


### 4.5 모델 학습


In [ ]:
def train_candidate_models(
    train_df: pd.DataFrame,
    feature_columns: list[str],
    regression_target: str,
    classification_target: str,
    random_state: int = 42,
) -> dict[str, Any]:
    """사용 가능한 Gradient Boosting 후보 모델을 모두 학습합니다.

    여기서는 best model을 고르지 않습니다. 모델 선택은 validation 평가 후에 수행합니다.
    """
    x_train = train_df[feature_columns]
    y_reg_train = train_df[regression_target]
    y_cls_train = train_df[classification_target]

    regression_candidates = build_regression_model_candidates(random_state=random_state)
    classification_candidates = build_classification_model_candidates(random_state=random_state)

    trained_regression_models = {}
    for model_name, model in regression_candidates.items():
        fitted_model = clone(model)
        fitted_model.fit(x_train, y_reg_train)
        trained_regression_models[model_name] = fitted_model

    trained_classification_models = {}
    for model_name, model in classification_candidates.items():
        fitted_model = clone(model)
        fitted_model.fit(x_train, y_cls_train)
        trained_classification_models[model_name] = fitted_model

    return {
        "regression_models": trained_regression_models,
        "classification_models": trained_classification_models,
        "feature_columns": feature_columns,
    }


### 4.6 모델 평가와 Threshold 탐색

회귀는 RMSE/MAE/R2/RMSLE 계열 지표를 보고, 분류는 Recall/Precision/F1/ROC-AUC/PR-AUC를 봅니다. 조류경보 예측은 미탐 위험이 크기 때문에 Recall을 중요하게 확인합니다.


In [ ]:
def inverse_transform_cells(pred_target: np.ndarray) -> np.ndarray:
    """회귀 예측값을 target 변환 방식에 맞춰 cells/mL 단위로 복원합니다."""
    pred_target = np.asarray(pred_target, dtype=float)

    if REGRESSION_TARGET_TRANSFORM == "log1p":
        return np.expm1(pred_target)
    if REGRESSION_TARGET_TRANSFORM == "log10_plus_1":
        return (10 ** pred_target) - 1
    if REGRESSION_TARGET_TRANSFORM == "none":
        return pred_target

    raise ValueError(f"지원하지 않는 REGRESSION_TARGET_TRANSFORM 값입니다: {REGRESSION_TARGET_TRANSFORM}")


def _safe_rmsle(y_true_cells: np.ndarray, y_pred_cells: np.ndarray) -> float | None:
    """실제값과 예측값이 모두 음수가 아닐 때만 RMSLE를 계산합니다."""
    y_true_cells = np.asarray(y_true_cells, dtype=float)
    y_pred_cells = np.asarray(y_pred_cells, dtype=float)

    if np.any(y_true_cells < 0) or np.any(y_pred_cells < 0):
        return None

    return float(np.sqrt(mean_squared_error(np.log1p(y_true_cells), np.log1p(y_pred_cells))))


def regression_metrics(y_true: pd.Series, y_pred: np.ndarray) -> dict[str, float | None]:
    """회귀 예측 성능을 평가합니다.

    MAE/RMSE/R2는 모델이 학습한 target 스케일에서 계산합니다.
    REGRESSION_TARGET_TRANSFORM을 알 수 있으면, 원 단위로 복원한 *_cells 지표도 함께 계산합니다.
    """
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    metrics: dict[str, float | None] = {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": rmse,
        "r2": float(r2_score(y_true, y_pred)),
    }

    y_true_cells = inverse_transform_cells(y_true.to_numpy())
    y_pred_cells = inverse_transform_cells(y_pred)
    metrics.update({
        "mae_cells": float(mean_absolute_error(y_true_cells, y_pred_cells)),
        "rmse_cells": float(np.sqrt(mean_squared_error(y_true_cells, y_pred_cells))),
        "rmsle_cells": _safe_rmsle(y_true_cells, y_pred_cells),
    })
    return metrics


def classification_metrics(
    y_true: pd.Series,
    risk_probability: np.ndarray,
    probability_threshold: float = 0.5,
) -> dict[str, Any]:
    y_pred = (risk_probability >= probability_threshold).astype(int)

    metrics = {
        "threshold": float(probability_threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
        "classification_report": classification_report(y_true, y_pred, zero_division=0, output_dict=True),
    }

    if len(pd.Series(y_true).dropna().unique()) == 2:
        metrics["roc_auc"] = float(roc_auc_score(y_true, risk_probability))
        metrics["pr_auc"] = float(average_precision_score(y_true, risk_probability))
    else:
        metrics["roc_auc"] = None
        metrics["pr_auc"] = None

    return metrics


def evaluate_threshold_candidates(
    y_true: pd.Series,
    risk_probability: np.ndarray,
    thresholds: list[float],
) -> pd.DataFrame:
    """기준값 후보별 정밀도/재현율/F1을 계산합니다.

    주의: threshold는 validation 기간에서만 탐색하세요.
    test 데이터에 threshold를 맞추면 운영 성능을 과대평가할 수 있습니다.
    """
    rows = []
    for threshold in thresholds:
        y_pred = (risk_probability >= threshold).astype(int)
        rows.append({
            "threshold": float(threshold),
            "precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "recall": float(recall_score(y_true, y_pred, zero_division=0)),
            "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        })
    return pd.DataFrame(rows)


def select_operational_threshold(
    threshold_df: pd.DataFrame,
    min_precision: float = 0.30,
) -> float:
    """최소 Precision 조건을 지키면서 Recall을 우선하는 threshold를 선택합니다."""
    if threshold_df.empty:
        return float(PROBABILITY_THRESHOLD)

    candidates = threshold_df[threshold_df["precision"].ge(min_precision)].copy()
    if candidates.empty:
        candidates = threshold_df.copy()

    selected = candidates.sort_values(
        ["recall", "f1", "precision"],
        ascending=[False, False, False],
    ).iloc[0]
    return float(selected["threshold"])


def evaluate_candidate_models(
    trained: dict[str, Any],
    valid_df: pd.DataFrame,
    regression_target: str,
    classification_target: str,
    probability_threshold: float = 0.5,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """모든 후보 모델을 평가하고 성능표와 threshold 후보표를 반환합니다."""
    feature_columns = trained["feature_columns"]
    x_valid = valid_df[feature_columns]

    metric_rows = []
    threshold_rows = []

    for model_name, model in trained["regression_models"].items():
        pred = model.predict(x_valid)
        metric = regression_metrics(valid_df[regression_target], pred)
        metric_rows.append({
            "task": "regression",
            "model_name": model_name,
            **metric,
        })

    for model_name, model in trained["classification_models"].items():
        risk_probability = model.predict_proba(x_valid)[:, 1]
        threshold_df = evaluate_threshold_candidates(
            valid_df[classification_target],
            risk_probability,
            THRESHOLD_CANDIDATES,
        )
        threshold_df.insert(0, "model_name", model_name)
        threshold_rows.append(threshold_df)

        selected_threshold = select_operational_threshold(threshold_df, MIN_PRECISION_FOR_THRESHOLD)
        metric = classification_metrics(valid_df[classification_target], risk_probability, selected_threshold)
        metric_rows.append({
            "task": "classification",
            "model_name": model_name,
            "selected_threshold": selected_threshold,
            **metric,
        })

    metrics_df = pd.DataFrame(metric_rows)
    threshold_result_df = pd.concat(threshold_rows, ignore_index=True) if threshold_rows else pd.DataFrame()
    return metrics_df, threshold_result_df


def select_best_models(
    trained: dict[str, Any],
    metrics_df: pd.DataFrame,
    regression_metric: str = "rmse",
    classification_metric: str = "recall",
) -> dict[str, Any]:
    """검증 성능표를 기준으로 최종 회귀 모델과 최종 분류 모델을 선택합니다."""
    reg_metrics = metrics_df[metrics_df["task"].eq("regression")].copy()
    cls_metrics = metrics_df[metrics_df["task"].eq("classification")].copy()

    if reg_metrics.empty:
        raise ValueError("회귀 모델 평가 지표가 없습니다.")
    if cls_metrics.empty:
        raise ValueError("분류 모델 평가 지표가 없습니다.")

    # 회귀 오차 지표는 낮을수록 좋습니다.
    best_reg_name = reg_metrics.sort_values(regression_metric, ascending=True).iloc[0]["model_name"]

    # 분류 지표는 높을수록 좋습니다.
    best_cls_row = cls_metrics.sort_values(classification_metric, ascending=False).iloc[0]
    best_cls_name = best_cls_row["model_name"]

    return {
        "regression_model": trained["regression_models"][best_reg_name],
        "classification_model": trained["classification_models"][best_cls_name],
        "feature_columns": trained["feature_columns"],
        "best_regression_model_name": best_reg_name,
        "best_classification_model_name": best_cls_name,
        "best_classification_threshold": float(best_cls_row.get("selected_threshold", PROBABILITY_THRESHOLD)),
        "candidate_metrics": metrics_df,
    }


### 4.7 모델과 지표 저장


In [ ]:
def _json_default(obj: Any) -> Any:
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if pd.isna(obj):
        return None
    return str(obj)


def save_models(
    trained: dict[str, Any],
    best_models: dict[str, Any],
    output_dir: Path,
    preprocessing_pipeline: Any | None = None,
) -> None:
    """모든 후보 모델, 선택된 best model, 비교 지표를 저장합니다."""
    output_dir.mkdir(parents=True, exist_ok=True)

    regression_dir = output_dir / "regression_candidates"
    classification_dir = output_dir / "classification_candidates"
    regression_dir.mkdir(exist_ok=True)
    classification_dir.mkdir(exist_ok=True)

    for model_name, model in trained["regression_models"].items():
        joblib.dump(model, regression_dir / f"{model_name}.joblib")

    for model_name, model in trained["classification_models"].items():
        joblib.dump(model, classification_dir / f"{model_name}.joblib")

    # 추론 편의성과 최종 산출물 계약을 위해 최종 모델 별칭을 저장합니다.
    joblib.dump(best_models["regression_model"], output_dir / REGRESSION_MODEL_FILE)
    joblib.dump(best_models["classification_model"], output_dir / CLASSIFICATION_MODEL_FILE)

    # 모델링 에이전트는 전처리를 만들지 않습니다.
    # 전처리팀이 학습 완료된 파이프라인을 제공하면 이 인자로 넘기고, 없으면 None을 저장합니다.
    joblib.dump(preprocessing_pipeline, output_dir / PREPROCESSING_PIPELINE_FILE)

    if "candidate_metrics" in best_models:
        best_models["candidate_metrics"].to_csv(output_dir / "candidate_model_metrics.csv", index=False)

    metadata = {
        "feature_columns": best_models["feature_columns"],
        "regression_target": REGRESSION_TARGET,
        "classification_target": CLASSIFICATION_TARGET,
        "probability_threshold": best_models.get("best_classification_threshold", PROBABILITY_THRESHOLD),
        "alert_cell_threshold": ALERT_CELL_THRESHOLD,
        "regression_target_transform": REGRESSION_TARGET_TRANSFORM,
        "best_regression_model_name": best_models.get("best_regression_model_name"),
        "best_classification_model_name": best_models.get("best_classification_model_name"),
        "regression_model_candidates": list(trained["regression_models"].keys()),
        "classification_model_candidates": list(trained["classification_models"].keys()),
        "model_selection_metric_regression": MODEL_SELECTION_METRIC_REGRESSION,
        "model_selection_metric_classification": MODEL_SELECTION_METRIC_CLASSIFICATION,
    }
    with open(output_dir / "model_metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2, default=_json_default)


def load_models(model_dir: Path) -> dict[str, Any]:
    """저장된 후보 모델과 best model alias를 불러옵니다."""
    with open(model_dir / "model_metadata.json", "r", encoding="utf-8") as f:
        metadata = json.load(f)

    regression_models = {}
    for model_name in metadata.get("regression_model_candidates", []):
        model_path = model_dir / "regression_candidates" / f"{model_name}.joblib"
        if model_path.exists():
            regression_models[model_name] = joblib.load(model_path)

    classification_models = {}
    for model_name in metadata.get("classification_model_candidates", []):
        model_path = model_dir / "classification_candidates" / f"{model_name}.joblib"
        if model_path.exists():
            classification_models[model_name] = joblib.load(model_path)

    return {
        "regression_model": joblib.load(model_dir / REGRESSION_MODEL_FILE),
        "classification_model": joblib.load(model_dir / CLASSIFICATION_MODEL_FILE),
        "preprocessing_pipeline": joblib.load(model_dir / PREPROCESSING_PIPELINE_FILE),
        "regression_models": regression_models,
        "classification_models": classification_models,
        "feature_columns": metadata["feature_columns"],
        "metadata": metadata,
    }


def save_metrics(metrics_df: pd.DataFrame, metric_dir: Path) -> None:
    metric_dir.mkdir(parents=True, exist_ok=True)

    regression_metrics_df = metrics_df[metrics_df["task"].eq("regression")].copy()
    classification_metrics_df = metrics_df[metrics_df["task"].eq("classification")].copy()

    regression_payload = regression_metrics_df.to_dict(orient="records")
    classification_payload = classification_metrics_df.to_dict(orient="records")

    with open(metric_dir / REGRESSION_METRIC_FILE, "w", encoding="utf-8") as f:
        json.dump(regression_payload, f, ensure_ascii=False, indent=2, default=_json_default)

    with open(metric_dir / CLASSIFICATION_METRIC_FILE, "w", encoding="utf-8") as f:
        json.dump(classification_payload, f, ensure_ascii=False, indent=2, default=_json_default)


def save_threshold_results(threshold_df: pd.DataFrame, metric_dir: Path) -> None:
    """분류 모델 검토용 threshold 후보표를 저장합니다."""
    metric_dir.mkdir(parents=True, exist_ok=True)
    threshold_df.to_csv(metric_dir / "classification_threshold_candidates.csv", index=False)


### 4.8 예측 결과 생성과 저장


In [ ]:
def predict_with_all_models(
    trained: dict[str, Any],
    input_df: pd.DataFrame,
    restore_cells: bool = True,
) -> pd.DataFrame:
    """모든 후보 모델의 예측 컬럼을 반환합니다."""
    feature_columns = trained["feature_columns"]
    x = input_df[feature_columns]

    output = input_df[[col for col in [DATE_COLUMN, SITE_COLUMN] if col in input_df.columns]].copy()

    for model_name, model in trained.get("regression_models", {}).items():
        pred_reg = model.predict(x)
        output[f"{model_name}_pred_regression_target"] = pred_reg

        if restore_cells:
            output[f"{model_name}_pred_cells"] = inverse_transform_cells(pred_reg)

    threshold = trained.get("metadata", {}).get("probability_threshold", PROBABILITY_THRESHOLD)
    for model_name, model in trained.get("classification_models", {}).items():
        pred_proba = model.predict_proba(x)[:, 1]
        output[f"{model_name}_alert_risk_probability"] = pred_proba
        output[f"{model_name}_alert_pred_label"] = (pred_proba >= threshold).astype(int)

    return output


def predict_with_best_models(
    trained: dict[str, Any],
    input_df: pd.DataFrame,
    restore_cells: bool = True,
) -> pd.DataFrame:
    """선택된 best model의 핵심 예측 컬럼만 반환합니다."""
    feature_columns = trained["feature_columns"]
    x = input_df[feature_columns]

    pred_reg = trained["regression_model"].predict(x)
    pred_proba = trained["classification_model"].predict_proba(x)[:, 1]
    threshold = trained.get("metadata", {}).get("probability_threshold", PROBABILITY_THRESHOLD)

    output = input_df[[col for col in [DATE_COLUMN, SITE_COLUMN] if col in input_df.columns]].copy()
    output["pred_regression_target"] = pred_reg

    if restore_cells:
        output["predicted_cells"] = inverse_transform_cells(pred_reg)

    output["alert_probability"] = pred_proba
    output["predicted_alert_label"] = (pred_proba >= threshold).astype(int)
    return output


def save_prediction_outputs(
    prediction_df: pd.DataFrame,
    original_df: pd.DataFrame,
    prediction_dir: Path,
    regression_target: str,
    classification_target: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """회귀/분류 예측 결과를 간결한 CSV 파일로 저장합니다."""
    prediction_dir.mkdir(parents=True, exist_ok=True)

    id_cols = [col for col in [DATE_COLUMN, SITE_COLUMN] if col in prediction_df.columns]

    regression_df = prediction_df[id_cols].copy()
    if regression_target in original_df.columns:
        regression_df["y_true_target"] = original_df[regression_target].to_numpy()
        regression_df["y_true_cells"] = inverse_transform_cells(original_df[regression_target].to_numpy())
    regression_df["y_pred_target"] = prediction_df["pred_regression_target"].to_numpy()
    regression_df["y_pred_cells"] = prediction_df["predicted_cells"].to_numpy()

    classification_df = prediction_df[id_cols].copy()
    if classification_target in original_df.columns:
        classification_df["y_true_alert"] = original_df[classification_target].to_numpy()
    classification_df["y_pred_alert"] = prediction_df["predicted_alert_label"].to_numpy()
    classification_df["y_pred_probability"] = prediction_df["alert_probability"].to_numpy()

    regression_df.to_csv(prediction_dir / REGRESSION_PREDICTION_FILE, index=False)
    classification_df.to_csv(prediction_dir / CLASSIFICATION_PREDICTION_FILE, index=False)
    return regression_df, classification_df


### 4.9 운영 보조 로직

공식 발령 자동화가 아니라 사전 점검 후보 표시용입니다.


In [ ]:
def apply_operational_alert_logic(
    prediction_df: pd.DataFrame,
    previous_cells_column: str = "previous_observed_cells",  # 수정 필요: 실제 컬럼명에 맞게 수정
    pred_cells_column: str = "predicted_cells",
    probability_column: str = "alert_probability",
    alert_cell_threshold: int = ALERT_CELL_THRESHOLD,
    probability_threshold: float = PROBABILITY_THRESHOLD,
) -> pd.DataFrame:
    # 참고:
    # 이 함수는 조류경보 공식 발령을 자동 결정하는 코드가 아닙니다.
    # 예측 결과를 바탕으로 관리자가 사전 점검할 수 있는
    # '관심 단계 후보'를 표시하는 운영 보조 로직입니다.
    output = prediction_df.copy()

    has_previous_cells = previous_cells_column in output.columns
    has_pred_cells = pred_cells_column in output.columns
    has_probability = probability_column in output.columns

    previous_exceeded = output[previous_cells_column] >= alert_cell_threshold if has_previous_cells else False
    predicted_cells_exceeded = output[pred_cells_column] >= alert_cell_threshold if has_pred_cells else False
    predicted_probability_exceeded = output[probability_column] >= probability_threshold if has_probability else False

    output["alert_candidate"] = previous_exceeded & (predicted_cells_exceeded | predicted_probability_exceeded)
    return output


### 4.10 SHAP / Feature Importance 설명


In [ ]:
def get_feature_importance_table(model: Any, feature_columns: list[str]) -> pd.DataFrame:
    if hasattr(model, "feature_importances_"):
        importance = model.feature_importances_
        return (
            pd.DataFrame({"feature": feature_columns, "importance": importance})
            .sort_values("importance", ascending=False)
            .reset_index(drop=True)
        )

    return pd.DataFrame(columns=["feature", "importance"])


def get_feature_importance_for_all_models(
    trained: dict[str, Any],
) -> pd.DataFrame:
    """변수 중요도 속성이 있는 모든 후보 모델의 중요도 표를 반환합니다."""
    rows = []
    feature_columns = trained["feature_columns"]

    for task, model_group_key in [
        ("regression", "regression_models"),
        ("classification", "classification_models"),
    ]:
        for model_name, model in trained.get(model_group_key, {}).items():
            importance_df = get_feature_importance_table(model, feature_columns)
            if importance_df.empty:
                rows.append({
                    "task": task,
                    "model_name": model_name,
                    "feature": None,
                    "importance": None,
                    "note": "이 모델은 feature_importances_ 속성을 제공하지 않습니다",
                })
                continue

            for _, row in importance_df.iterrows():
                rows.append({
                    "task": task,
                    "model_name": model_name,
                    "feature": row["feature"],
                    "importance": row["importance"],
                    "note": None,
                })

    return pd.DataFrame(rows)


def _extract_2d_shap_values(shap_values: Any) -> np.ndarray:
    values = np.asarray(shap_values.values)
    # 일부 이진 분류 모델은 (샘플 수, 입력 변수 수, 클래스 수) 형태의 SHAP 값을 반환할 수 있습니다.
    # 이 경우 관례적으로 양성 위험 클래스의 SHAP 값을 사용합니다.
    if values.ndim == 3:
        values = values[:, :, -1]
    return values


def _prediction_value_for_row(
    prediction_df: pd.DataFrame | None,
    row_idx: int,
    model_name: str,
    task: str,
) -> dict[str, Any]:
    if prediction_df is None:
        return {}

    result: dict[str, Any] = {}
    if task == "classification":
        proba_col = f"{model_name}_alert_risk_probability"
        label_col = f"{model_name}_alert_pred_label"
        if proba_col in prediction_df.columns:
            result["alert_probability"] = prediction_df.iloc[row_idx][proba_col]
        if label_col in prediction_df.columns:
            result["predicted_alert_label"] = prediction_df.iloc[row_idx][label_col]
    elif task == "regression":
        pred_col = f"{model_name}_pred_cells"
        if pred_col in prediction_df.columns:
            result["predicted_cells"] = prediction_df.iloc[row_idx][pred_col]
    return result


def make_shap_summary_table_for_all_models(
    trained: dict[str, Any],
    input_df: pd.DataFrame,
    prediction_df: pd.DataFrame | None = None,
    task: str = "classification",
    top_n: int = 3,
) -> pd.DataFrame:
    """모든 후보 모델에 대해 행별 상위 SHAP feature를 반환합니다.

    이 함수는 SHAP 설명 결과의 구조를 정의합니다.
    실제 SHAP 값은 실제 데이터로 코드를 실행했을 때 계산됩니다.
    """
    try:
        import shap
    except ImportError as exc:
        raise ImportError("shap이 설치되어 있지 않습니다. shap을 설치하거나 feature importance 기반 대체 결과를 사용하세요.") from exc

    if task == "classification":
        model_group = trained.get("classification_models", {})
    elif task == "regression":
        model_group = trained.get("regression_models", {})
    else:
        raise ValueError("task는 'classification' 또는 'regression'이어야 합니다.")

    feature_columns = trained["feature_columns"]
    x = input_df[feature_columns]
    all_rows = []

    for model_name, model in model_group.items():
        explainer = shap.Explainer(model, x)
        shap_values = explainer(x)
        values_2d = _extract_2d_shap_values(shap_values)

        for row_idx in range(len(x)):
            row_values = values_2d[row_idx]
            top_indices = np.argsort(np.abs(row_values))[::-1][:top_n]

            row = {"task": task, "model_name": model_name}
            for col in [DATE_COLUMN, SITE_COLUMN]:
                if col in input_df.columns:
                    row[col] = input_df.iloc[row_idx][col]

            row.update(_prediction_value_for_row(prediction_df, row_idx, model_name, task))

            for rank, feature_idx in enumerate(top_indices, start=1):
                row[f"top_{rank}_feature"] = feature_columns[feature_idx]
                row[f"top_{rank}_shap_value"] = row_values[feature_idx]
            all_rows.append(row)

    return pd.DataFrame(all_rows)


def make_shap_top_reasons_table(
    trained: dict[str, Any],
    input_df: pd.DataFrame,
    best_prediction_df: pd.DataFrame,
    top_n: int = 3,
) -> pd.DataFrame:
    """선택된 best 분류 모델 기준으로 필수 SHAP 상위 원인표를 만듭니다."""
    try:
        import shap
    except ImportError:
        return make_feature_importance_fallback_reasons(trained, input_df, best_prediction_df, top_n=top_n)

    feature_columns = trained["feature_columns"]
    x = input_df[feature_columns]
    model = trained["classification_model"]

    explainer = shap.Explainer(model, x)
    shap_values = explainer(x)
    values_2d = _extract_2d_shap_values(shap_values)

    rows = []
    for row_idx in range(len(x)):
        row_values = values_2d[row_idx]
        top_indices = np.argsort(np.abs(row_values))[::-1][:top_n]

        row = {}
        for col in [DATE_COLUMN, SITE_COLUMN]:
            if col in input_df.columns:
                row[col] = input_df.iloc[row_idx][col]

        row["predicted_cells"] = best_prediction_df.iloc[row_idx].get("predicted_cells")
        row["alert_probability"] = best_prediction_df.iloc[row_idx].get("alert_probability")
        row["predicted_alert_label"] = best_prediction_df.iloc[row_idx].get("predicted_alert_label")

        for rank, feature_idx in enumerate(top_indices, start=1):
            row[f"shap_top_{rank}"] = feature_columns[feature_idx]
            row[f"shap_top_{rank}_value"] = row_values[feature_idx]
        rows.append(row)

    return pd.DataFrame(rows)


def make_feature_importance_fallback_reasons(
    trained: dict[str, Any],
    input_df: pd.DataFrame,
    best_prediction_df: pd.DataFrame,
    top_n: int = 3,
) -> pd.DataFrame:
    """SHAP을 사용할 수 없을 때 변수 중요도 기반 대체 원인표를 만듭니다.

    각 예측 행에 동일한 전역 변수 중요도를 붙입니다.
    행별 SHAP보다 설명력은 약하지만, 다음 에이전트가 기대하는 출력 구조는 유지합니다.
    """
    importance_df = get_feature_importance_table(trained["classification_model"], trained["feature_columns"])
    top_features = importance_df.head(top_n).to_dict(orient="records")

    rows = []
    for row_idx in range(len(input_df)):
        row = {}
        for col in [DATE_COLUMN, SITE_COLUMN]:
            if col in input_df.columns:
                row[col] = input_df.iloc[row_idx][col]

        row["predicted_cells"] = best_prediction_df.iloc[row_idx].get("predicted_cells")
        row["alert_probability"] = best_prediction_df.iloc[row_idx].get("alert_probability")
        row["predicted_alert_label"] = best_prediction_df.iloc[row_idx].get("predicted_alert_label")

        for rank in range(1, top_n + 1):
            if rank <= len(top_features):
                row[f"shap_top_{rank}"] = top_features[rank - 1].get("feature")
                row[f"shap_top_{rank}_value"] = top_features[rank - 1].get("importance")
            else:
                row[f"shap_top_{rank}"] = None
                row[f"shap_top_{rank}_value"] = None
        rows.append(row)

    return pd.DataFrame(rows)


def save_explain_outputs(
    importance_df: pd.DataFrame,
    shap_top_reasons_df: pd.DataFrame,
    explain_dir: Path,
) -> None:
    explain_dir.mkdir(parents=True, exist_ok=True)
    importance_df.to_csv(explain_dir / FEATURE_IMPORTANCE_FILE, index=False)
    shap_top_reasons_df.to_csv(explain_dir / SHAP_TOP_REASONS_FILE, index=False)


### 4.11 시나리오 후처리

예측값과 SHAP 상위 feature를 바탕으로 위험 유형을 분류합니다. 이 단계는 새 모델이 아니라 rule-based post-processing입니다.


In [ ]:
def _feature_matches_group(feature_name: Any, group_features: list[str]) -> bool:
    if feature_name is None or pd.isna(feature_name):
        return False

    feature_name_lower = str(feature_name).lower()
    return any(group_feature.lower() in feature_name_lower for group_feature in group_features)


def _count_top_feature_groups(top_features: list[Any]) -> dict[str, int]:
    counts = {group_name: 0 for group_name in SCENARIO_FEATURE_GROUPS}
    for feature_name in top_features:
        for group_name, group_features in SCENARIO_FEATURE_GROUPS.items():
            if _feature_matches_group(feature_name, group_features):
                counts[group_name] += 1
    return counts


def classify_scenario(row: pd.Series) -> tuple[str, str, str]:
    """위험 확률과 상위 SHAP feature를 바탕으로 rule-based 시나리오를 분류합니다."""
    top_features = [row.get("shap_top_1"), row.get("shap_top_2"), row.get("shap_top_3")]
    group_counts = _count_top_feature_groups(top_features)
    active_groups = [group_name for group_name, count in group_counts.items() if count > 0]
    probability = float(row.get("alert_probability", 0.0) or 0.0)

    if probability >= HIGH_RISK_THRESHOLD and len(active_groups) >= 2:
        scenario_type = "복합 고위험 시나리오"
    elif group_counts["stagnation"] >= 2:
        scenario_type = "수체 정체·성층 위험 시나리오"
    elif group_counts["rainfall_inflow"] >= 2:
        scenario_type = "강우 이후 영양염류 유입 시나리오"
    elif group_counts["growth_condition"] >= 2:
        scenario_type = "고온·고일사 성장 촉진 시나리오"
    elif group_counts["past_bloom"] >= 2:
        scenario_type = "과거 증식 추세 지속 시나리오"
    elif probability >= MEDIUM_RISK_THRESHOLD:
        scenario_type = "관찰 강화 시나리오"
    else:
        scenario_type = "일반 안정 시나리오"

    scenario_reason = (
        f"alert_probability={probability:.3f}, "
        f"top_features={[str(feature) for feature in top_features if pd.notna(feature)]}, "
        f"matched_groups={group_counts}"
    )
    action_category = SCENARIO_ACTION_CATEGORY.get(scenario_type, "모니터링 강화")
    return scenario_type, scenario_reason, action_category


def build_scenario_results(shap_top_reasons_df: pd.DataFrame) -> pd.DataFrame:
    """시나리오 유형, 시나리오 근거, 권장 대응 범주 컬럼을 추가합니다."""
    required = ["predicted_cells", "alert_probability", "predicted_alert_label"]
    missing = [col for col in required if col not in shap_top_reasons_df.columns]
    if missing:
        raise KeyError(f"시나리오 입력에 필수 컬럼이 없습니다: {missing}")

    output = shap_top_reasons_df.copy()
    scenario_values = output.apply(classify_scenario, axis=1, result_type="expand")
    output["scenario_type"] = scenario_values[0]
    output["scenario_reason"] = scenario_values[1]
    output["recommended_action_category"] = scenario_values[2]

    keep_cols = [
        DATE_COLUMN,
        SITE_COLUMN,
        "predicted_cells",
        "alert_probability",
        "predicted_alert_label",
        "shap_top_1",
        "shap_top_2",
        "shap_top_3",
        "scenario_type",
        "scenario_reason",
        "recommended_action_category",
    ]
    existing_cols = [col for col in keep_cols if col in output.columns]
    return output[existing_cols]


def build_scenario_input_for_llm(scenario_results_df: pd.DataFrame) -> list[dict[str, Any]]:
    """다음 에이전트가 사용할 구조화 JSON 객체를 만듭니다.

    이 함수는 브리핑 문장을 만들지 않습니다.
    다음 LLM Briefing Agent가 필요한 구조화 데이터만 만듭니다.
    """
    records: list[dict[str, Any]] = []
    for _, row in scenario_results_df.iterrows():
        top_reasons = []
        for rank in range(1, 4):
            feature = row.get(f"shap_top_{rank}")
            if pd.notna(feature):
                top_reasons.append({"rank": rank, "feature": feature})

        records.append({
            "date": str(row.get(DATE_COLUMN)) if DATE_COLUMN in row else None,
            "site": row.get(SITE_COLUMN) if SITE_COLUMN in row else None,
            "forecast_horizon": FORECAST_HORIZON,
            "predicted_cells": row.get("predicted_cells"),
            "alert_probability": row.get("alert_probability"),
            "predicted_alert_label": row.get("predicted_alert_label"),
            "top_reasons": top_reasons,
            "scenario_type": row.get("scenario_type"),
            "scenario_reason": row.get("scenario_reason"),
            "recommended_action_category": row.get("recommended_action_category"),
        })
    return records


def save_scenario_outputs(scenario_results_df: pd.DataFrame, scenario_dir: Path) -> list[dict[str, Any]]:
    scenario_dir.mkdir(parents=True, exist_ok=True)
    scenario_results_df.to_csv(scenario_dir / SCENARIO_RESULT_FILE, index=False)

    llm_input = build_scenario_input_for_llm(scenario_results_df)
    with open(scenario_dir / SCENARIO_LLM_INPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(llm_input, f, ensure_ascii=False, indent=2, default=_json_default)
    return llm_input


## 5. 실행 파이프라인

아래부터는 실제 실행 셀입니다. 전처리팀의 최종 입력 파일과 config 컬럼명이 맞는 상태에서 위에서부터 순서대로 실행합니다.


### 5.1 데이터 로드


In [ ]:
df = load_model_input(MODEL_INPUT_PATH)
df.head()


### 5.2 시간 기준 Train / Validation 분리


In [ ]:
train_df, valid_df = time_based_train_valid_split(
    df,
    DATE_COLUMN,
    SPLIT_COLUMN,
    VALID_SIZE_RATIO,
)

train_df.shape, valid_df.shape


### 5.3 Feature / Target 분리


In [ ]:
feature_columns = get_feature_columns(
    train_df,
    DROP_COLUMNS,
    SCHEMA_PATH,
)

x_train, y_reg_train, y_cls_train = split_features_targets(
    train_df,
    feature_columns,
    REGRESSION_TARGET,
    CLASSIFICATION_TARGET,
)

x_valid, y_reg_valid, y_cls_valid = split_features_targets(
    valid_df,
    feature_columns,
    REGRESSION_TARGET,
    CLASSIFICATION_TARGET,
)

len(feature_columns), feature_columns[:10]


### 5.4 후보 모델 학습


In [ ]:
trained = train_candidate_models(
    train_df,
    feature_columns,
    REGRESSION_TARGET,
    CLASSIFICATION_TARGET,
    RANDOM_STATE,
)

trained.keys()


### 5.5 후보 모델 평가 및 Best Model 선택


In [ ]:
metrics_df, threshold_df = evaluate_candidate_models(
    trained,
    valid_df,
    REGRESSION_TARGET,
    CLASSIFICATION_TARGET,
    PROBABILITY_THRESHOLD,
)

best_models = select_best_models(
    trained,
    metrics_df,
    MODEL_SELECTION_METRIC_REGRESSION,
    MODEL_SELECTION_METRIC_CLASSIFICATION,
)

metrics_df


### 5.6 모델과 평가 결과 저장


In [ ]:
save_models(
    trained,
    best_models,
    OUTPUT_DIR,
)

save_metrics(metrics_df, METRIC_DIR)
save_threshold_results(threshold_df, METRIC_DIR)

loaded = load_models(OUTPUT_DIR)
loaded["metadata"]


### 5.7 예측 결과 생성 및 저장


In [ ]:
all_model_prediction_df = predict_with_all_models(
    loaded,
    valid_df,
)

best_model_prediction_df = predict_with_best_models(
    loaded,
    valid_df,
)

regression_prediction_df, classification_prediction_df = save_prediction_outputs(
    best_model_prediction_df,
    valid_df,
    PREDICTION_DIR,
    REGRESSION_TARGET,
    CLASSIFICATION_TARGET,
)

best_model_prediction_df.head()


### 5.8 SHAP 또는 Feature Importance 기반 원인 추출


In [ ]:
importance_df = get_feature_importance_for_all_models(loaded)

try:
    all_model_shap_cls_df = make_shap_summary_table_for_all_models(
        loaded,
        valid_df,
        all_model_prediction_df,
        task="classification",
    )

    all_model_shap_reg_df = make_shap_summary_table_for_all_models(
        loaded,
        valid_df,
        all_model_prediction_df,
        task="regression",
    )
except ImportError:
    all_model_shap_cls_df = pd.DataFrame()
    all_model_shap_reg_df = pd.DataFrame()

shap_top_reasons_df = make_shap_top_reasons_table(
    loaded,
    valid_df,
    best_model_prediction_df,
)

save_explain_outputs(
    importance_df,
    shap_top_reasons_df,
    EXPLAIN_DIR,
)

shap_top_reasons_df.head()


### 5.9 시나리오 결과 생성 및 다음 에이전트용 JSON 저장


In [ ]:
scenario_results_df = build_scenario_results(shap_top_reasons_df)
scenario_input_for_llm = save_scenario_outputs(
    scenario_results_df,
    SCENARIO_DIR,
)

scenario_results_df.head()


## 6. 최종 산출물

실행 후 아래 파일이 생성됩니다.

```text
artifacts/predictions/regression_predictions.csv
artifacts/predictions/classification_predictions.csv
artifacts/metrics/regression_metrics.json
artifacts/metrics/classification_metrics.json
artifacts/metrics/classification_threshold_candidates.csv
artifacts/explain/feature_importance.csv
artifacts/explain/shap_top_reasons.csv
artifacts/scenario/scenario_results.csv
artifacts/scenario/scenario_input_for_llm.json
artifacts/models/regression_model.pkl
artifacts/models/classification_model.pkl
artifacts/models/preprocessing_pipeline.pkl
```

주의:

- 성능 수치는 실제 데이터로 실행한 뒤 확인합니다.
- threshold는 validation 기준으로만 탐색해야 합니다.
- 시나리오 결과는 대응 유형을 구조화하는 것이며, 행정 명령을 자동 생성하지 않습니다.
